# Algorithm Documentation

This notebook documents every algorithm implemented in this project, organised into three groups:

1. **Baseline algorithms** — standard KNN and its direct extensions
2. **Adaptive-k algorithms** — per-query k selection strategies (early exploration)
3. **FairRank family** — the main contribution: rank-correction for class imbalance

For each algorithm the document covers:
- What problem it solves
- How it works (mechanics and key equations)
- Design decisions and parameters
- **Empirical results**, for algorithms that appear in the final benchmark

Results come directly from `entrega_final.ipynb` (5-rep × 10-fold CV, 40 datasets) and `meta_analysis.ipynb` (LODO-CV instance space). Algorithms that did not make it into the final benchmark have no results section.

---

## 0. The Class-Imbalance Problem

Standard KNN votes by majority among the $k$ nearest neighbours. When the training set has far more majority-class points than minority-class points (imbalance ratio $r = N_{maj} / N_{min} \gg 1$), two things go wrong:

1. **Vote dilution.** Even near the true decision boundary, the $k$ neighbours are almost certainly dominated by majority points — just because there are more of them.
2. **Sampling bias in distances.** The expected minimum distance to the majority class is smaller than to the minority class, even when both classes have identical underlying densities. This is a pure statistical artefact from having more samples.

All algorithms in this project address one or both of these problems.

**Key tension that runs through all results:** G-Mean rewards algorithms that identify minority cases aggressively (high recall). MCC and F1 also penalise false positives, so they reward precision. An algorithm that predicts minority too eagerly wins G-Mean but loses MCC. This trade-off is the central thread of the empirical analysis.

**Final benchmark** (`entrega_final.ipynb §6`): 5-rep × 10-fold stratified CV across 40 datasets. Primary metric: G-Mean. Secondary: MCC, F1, ROC-AUC, PR-AUC. Statistical framework: Friedman + Wilcoxon with Holm correction (Demšar 2006).

---

## 1. Baseline Algorithms

### 1.1 KNNClassifier (Standard KNN)

**File:** `src/algorithms/baseline/knn_base.py`  
**Classes:** `KNNBase`, `KNNClassifier`, `KNNClassifierFast`

#### How it works

The standard $k$-Nearest Neighbours classifier:

1. Store all training points $\{(x_i, y_i)\}_{i=1}^N$.
2. For a query $x$, compute the distance $d(x, x_i)$ to every training point.
3. Take the $k$ points with the smallest distances — the $k$ nearest neighbours.
4. Return the most frequent class label among those $k$ neighbours.

$$\hat{y}(x) = \arg\max_{c} \sum_{i=1}^{k} \mathbf{1}[y_{(i)} = c]$$

$$P(y = c \mid x) = \frac{1}{k} \sum_{i=1}^{k} \mathbf{1}[y_{(i)} = c]$$

#### Class hierarchy

| Class | Description |
|---|---|
| `KNNBase` | Abstract base — stores data, sorts neighbours, delegates `aggregate()` |
| `KNNClassifier` | Implements `aggregate()` via `Counter.most_common(1)` |
| `KNNClassifierFast` | Same predictions; uses `scipy.cdist` for vectorised distance computation |

`KNNClassifierFast` replaces the Python loop with a single C-level matrix call — results are bit-for-bit identical, only speed differs. It is the base class for all other variants.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k` | 5 | Number of neighbours (`0` = use all) |
| `distance_func` | Euclidean | Any `scipy.spatial.distance` function |
| `n_jobs` | 1 | Parallelism via `joblib` |

#### Note on results

`KNNClassifier` with a fixed $k$ is not benchmarked in the final run — `KNNOptK` (below) is the standard-KNN representative, since selecting $k$ by CV is the correct baseline.

### 1.2 KNNOptK — KNN with Cross-Validated k

**File:** `src/algorithms/baseline/knn_base.py`  
**Class:** `KNNOptK`

#### How it works

Selects the best $k$ for each training set via inner stratified cross-validation:

1. Build a candidate set of odd $k$ values: $\{1, 3, 5, \ldots, \lfloor\sqrt{N_{train}}\rfloor\}$.
2. For each candidate $k$, score by balanced accuracy on each CV fold.
3. Select $k^* = \arg\max_k \overline{\text{BA}}_k$.
4. Fit a final `KNNClassifierFast(k=k^*)` on the full training set.

Balanced accuracy is used (not plain accuracy) because it weights each class equally.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_max` | `floor(sqrt(N))` | Upper bound on $k$ candidates |
| `inner_cv_folds` | from config | Folds for inner CV |
| `n_jobs` | 8 | Parallel $(k, \text{fold})$ evaluation |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§9`*

KNNOptK produces the most **counterintuitive result** in the entire benchmark. As stated in `entrega_final §11.1`:

> *"KNNFairRankJointCV ranks #1 (mean G-Mean = 0.796 vs **0.631 for KNNOptK**)."*

| Metric | KNNOptK | Rank (out of 10) |
|---|---|---|
| G-Mean | 0.673 | **10th (last)** |
| MCC | 0.556 | 3rd |
| F1 | 0.586 | 4th |
| ROC-AUC | 0.803 | last |

It is **last on G-Mean** yet **3rd on MCC**. The instance space analysis (`entrega_final §9.2`) reveals why:

> *"KNNOptK wins MCC on 17/40 datasets (42%) despite ranking last on G-Mean. Under heavy class imbalance, predicting mostly majority yields a large TN that inflates MCC even when minority sensitivity is poor."*

The inner CV selection of $k$ by balanced accuracy is not enough to fix vote dilution. The selected $k$ tends to be small and conservative — the algorithm classifies most points as majority, achieving high specificity but very low sensitivity. This produces good MCC and F1 but catastrophic G-Mean.

The stress test across IR quartiles (`entrega_final §9.3`) confirms this is not a mild gap: KNNOptK achieves G-Mean 0.553 at mild imbalance and 0.565 at severe imbalance, while the FairRank variants sit consistently above 0.75. Every FairRank variant beats KNNOptK on G-Mean with statistical significance ($p < 10^{-5}$, Holm-corrected), and all beat it on ROC-AUC.

### 1.3 KNNWeighted — Class-Frequency Weighting

**File:** `src/algorithms/baseline/knn_weighted.py`  
**Class:** `KNNWeighted`

#### How it works

Multiplies each class's raw vote count by an inverse-frequency weight before deciding the winner. The weight follows the sklearn `"balanced"` scheme:

$$w_c = \frac{N_{total}}{n_{classes} \cdot N_c}$$

For binary classification: $w_{min} = r,\ w_{maj} = 1$. The weighted probability estimate is:

$$\tilde{P}(y = c \mid x) = \frac{w_c \cdot \text{count}_c(x)}{\sum_{c'} w_{c'} \cdot \text{count}_{c'}(x)}$$

#### Parameters

Inherits all parameters from `KNNClassifierFast`. No additional parameters.

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§9`*

KNNWeighted is the **sanity-check** baseline: if the main FairRank variants cannot beat it, the rank-correction mechanism adds nothing beyond frequency reweighting.

| Metric | KNNWeighted | Rank |
|---|---|---|
| G-Mean | 0.797 | 7th |
| MCC | 0.534 | 9th |
| F1 | 0.583 | 5th |
| ROC-AUC | 0.841 | 8th |

It beats KNNOptK clearly on G-Mean (+0.124) — confirming vote reweighting helps minority recall — but falls near the bottom on MCC, meaning the fixed factor-$r$ amplification overshoots and creates too many false positives.

The most revealing finding comes from the stress test (`entrega_final §9.3`):

> *"KNNWeighted degrades as IR increases. It wins 20% of datasets in Q1 (mild imbalance) but **0% in Q3 and Q4**."*

This is the key weakness: the fixed weight $w_{min} = r$ is a global constant. As $r$ grows, the weight grows too — the algorithm over-corrects in majority-dense regions, producing more and more false positives. The FairRank correction is also proportional to $r$, but is applied via a *comparison* mechanism rather than amplifying vote counts directly, which makes it more resistant to this degradation.

In the **degenerate dataset regime** (9 datasets with extremely few minority samples, `entrega_final §8`), KNNWeighted's picture reverses dramatically:

> *"KNNWeighted ranks first (avg rank 3.56)" on G-Mean in the degenerate subset.*

When there are so few minority samples that inner CV is unreliable, the simple reweighting strategy that requires no optimisation outperforms the tuning-dependent FairRank variants.

### 1.4 DANN — Discriminant Adaptive Nearest Neighbours

**File:** `src/algorithms/baseline/dann.py`  
**Class:** `DANN`  
**Reference:** Hastie & Tibshirani (1996)

#### How it works

Adapts the distance metric *locally* at each query point so that it stretches along discriminant directions and shrinks along within-class variation.

For each query $x$:
1. Find the 50 nearest Euclidean neighbours (local neighbourhood).
2. Compute within-class scatter $W = \sum_c \sum_{i \in \text{local}, y_i=c} (x_i-\mu_c)(x_i-\mu_c)^\top$.
3. Compute between-class scatter $B = \sum_c N_c (\mu_c - \mu)(\mu_c - \mu)^\top$.
4. Regularise: $W^* = W + \sigma I$.
5. Local metric: $M(x) = (W^*)^{-1} B (W^*)^{-1}$.
6. Adapted distance: $d_M(x, x_i) = \sqrt{(x-x_i)^\top M(x)(x-x_i)}$.
7. Classify by majority vote among $k$ nearest neighbours under $d_M$.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k` | 5 | Neighbours for the final vote |
| `sigma` | 1.0 | Regularisation |
| `metric` | `"euclidean"` | Base metric for the initial neighbourhood |

#### Note on results

DANN is not included in the final benchmark in `entrega_final.ipynb`. It was explored in earlier phases but dropped, primarily because:
1. A per-query $p \times p$ matrix inversion is too expensive for a 40-dataset, 5-rep benchmark.
2. The adapted metric helps with boundary geometry but does not correct the vote dilution problem — the final vote among $k$ neighbours still suffers from the same structural imbalance bias.

---

## 2. Adaptive-k Algorithms

These algorithms select $k$ per query point rather than fixing it globally. They represent an early design direction that was eventually superseded by the FairRank approach.

### 2.1 KNNAdaptiveEntropy

**File:** `src/algorithms/adaptive_k/knn_adaptive_entropy.py`  
**Class:** `KNNAdaptiveEntropy`

#### How it works

For each query $x$, select $k$ by maximising the Shannon entropy of the neighbourhood class distribution:

$$H(k) = -\sum_c p_c(k) \log_2 p_c(k)$$

**Intuition:** High entropy = balanced mix = near the decision boundary. Maximising $H(k)$ finds the $k$ at which the neighbourhood is most informationally rich about the local class structure.

Rather than evaluating all $k$ candidates, a halve/double hill-climb is used — $O(\log N_{train})$ evaluations per query:

```
k_start = floor(sqrt(N_train))   # odd
halve downward while H improves
double upward while H improves
```

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_min` | 1 | Minimum $k$ |
| `k_max` | `floor(sqrt(N))` | Maximum $k$ |
| `smoothing` | 1e-9 | Prevents $\log 0$ |

#### Note on results

Not in the final benchmark. Maximising entropy is not equivalent to correcting for imbalance: on a strongly imbalanced dataset, entropy is maximised with a very large $k$ that includes enough minority samples to raise the proportion — not through a principled distance correction. The FairRank approach addresses the root cause directly.

### 2.2 DANNAdaptive — DANN with Adaptive k

**File:** `src/algorithms/baseline/dann_adaptive.py`  
**Class:** `DANNAdaptive`

#### How it works

Combines DANN's locally adapted metric with per-query $k$ selection via the same halve/double hill-climb. Two strategies for the $k$ criterion:

- **`"entropy"`** — maximise Shannon entropy $H(k)$ (same as KNNAdaptiveEntropy)
- **`"eigen"`** — maximise effective dimensionality: number of PCA components explaining ≥95% of neighbourhood variance

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_min` | 1 | Minimum $k$ |
| `k_max` | `floor(sqrt(N))` | Maximum $k$ |
| `sigma` | 1.0 | DANN regularisation |
| `adaptation_strategy` | `"entropy"` | `"entropy"` or `"eigen"` |

#### Note on results

Not in the final benchmark. Inherits both DANN's cost (matrix inversion per query) and the entropy-maximisation limitation. Not a priority for the final comparison.

---

## 3. The FairRank Family

This is the main contribution of the project.

### 3.0 The Core Insight

Under a homogeneous Poisson process model, the expected distance to the $k$-th nearest neighbour of class $c$ scales as:

$$E[d_k^c] \propto \left(\frac{k}{N_c}\right)^{1/d}$$

Setting $E[d_{k_{eff}}^{maj}] = E[d_1^{min}]$ and solving:

$$\left(\frac{k_{eff}}{N_{maj}}\right)^{1/d} = \left(\frac{1}{N_{min}}\right)^{1/d} \implies \boxed{k_{eff} = r}$$

The exponent $1/d$ cancels — the fair rank is **dimension-free**. The $k_{eff}$-th majority neighbour is statistically equivalent to the 1st minority neighbour, purely as a sampling artefact. The algorithm corrects for this by comparing minority rank $i$ to majority rank $i \cdot k_{eff}$ instead of rank $i$.

---

### 3.1 KNNFairRank (v3 — Core Algorithm)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank.py`  
**Class:** `KNNFairRank`

#### How it works

**At fit time:**
1. Identify minority/majority class by count. Compute $r = N_{maj} / N_{min}$.
2. Store separate arrays $X_{min}$, $X_{maj}$ for per-class lookups.
3. Size the majority neighbourhood: $k_{maj} = \max(\lceil r \rceil \cdot n_{votes} + \text{buffer},\ k_{floor})$, capped at $k_{cap}$.

**At predict time (query $x$):**
1. Sorted minority distances: $d_1^{min} \leq d_2^{min} \leq \ldots \leq d_{k_{min}}^{min}$
2. Sorted majority distances: $d_1^{maj} \leq d_2^{maj} \leq \ldots \leq d_{k_{maj}}^{maj}$
3. Set $k_{eff} = \text{round}(r)$.
4. Run $n_{votes}$ rank-corrected binary comparisons:
$$\text{vote}_i = \mathbf{1}\left[d_i^{min} < d_{i \cdot k_{eff}}^{maj}\right], \quad i = 1, \ldots, n_{votes}$$
5. Predict minority if vote fraction $> 0.5$, else majority.

Using per-class neighbour lists (not a joint $k$-NN list) ensures both classes are always represented regardless of imbalance.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_min` | 10 | Minority neighbours fetched |
| `k_maj_buffer` | 10 | Extra majority neighbours beyond the theoretical minimum |
| `k_maj_floor` | 30 | Minimum majority neighbourhood size |
| `k_maj_cap` | 1000 | Cap on majority neighbourhood size |
| `n_votes` | 5 | Number of rank-corrected comparisons |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§9`*

KNNFairRank (v3) demonstrates the core benefit of the rank correction — and also its main weakness when left untuned:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.819 | 6th |
| MCC | 0.503 | **10th (last)** |
| ROC-AUC | 0.863 | 5th |

The G-Mean gap over KNNOptK is dramatic and consistent across all imbalance levels — going from 0.819 vs 0.673 on average, and at severe imbalance (Q1) from 0.765 vs 0.565. This validates the theoretical correction: the per-class comparison with $k_{eff} = r$ correctly identifies minority cases that the standard vote would miss.

However, the fixed $k_{eff} = r$ is **too aggressive**: the theoretical derivation assumes homogeneous Poisson density, which fails in majority-dense regions where the minority is naturally scarce. In those regions, $k_{eff} = r$ over-corrects — the majority reference distance $d_{i \cdot k_{eff}}^{maj}$ is farther than it should be, causing the algorithm to predict minority when it shouldn't. This inflates false positives, tanking MCC and F1.

The base v3 is best understood as the **anchor** for all variants: it sets the maximum possible G-Mean correction and the minimum possible MCC. Every subsequent variant attempts to recover some MCC by softening or adapting the correction.

### 3.2 KNNFairRankMagnitude (Modification B)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank_b.py`  
**Class:** `KNNFairRankMagnitude`

#### How it works

Replaces the binary vote $\mathbf{1}[d_i^{min} < d_{i \cdot k_{eff}}^{maj}]$ with a continuous confidence score:

$$s_i = \frac{d_{i \cdot k_{eff}}^{maj}}{d_i^{min} + d_{i \cdot k_{eff}}^{maj}} \in [0, 1]$$

- $s_i \to 1$: majority reference is much farther — strong minority signal
- $s_i = 0.5$: distances equal — no signal
- $s_i \to 0$: minority is much farther — strong majority signal

The mean $\bar{s}$ replaces the binary vote fraction. Hard decision: predict minority iff $\bar{s} > 0.5$.

#### Note on results

Not included in the final 10-algorithm benchmark in `entrega_final.ipynb`. Its main contribution — better-calibrated soft scores — is partly absorbed by KNNFairRankJackknife and KNNFairRankEnsemble, which also produce smoother probability estimates through averaging.

### 3.3 KNNFairRankCV (Modification C)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank_c.py`  
**Class:** `KNNFairRankCV`

#### How it works

Introduces an exponent $\alpha$ to tune the correction strength:

$$k_{eff} = \text{round}(r^\alpha)$$

| $\alpha$ | $k_{eff}$ | Effect |
|---|---|---|
| 0 | 1 | No rank correction |
| 0.5 | $\sqrt{r}$ | Square-root damping |
| 1 | $r$ | Full v3 correction |

$\alpha$ is selected from $\{0.25, 0.5, 0.75, 1.0\}$ by inner stratified CV optimising a configurable metric (default: G-Mean). Available metrics: `geometric_mean`, `balanced_accuracy`, `f1`, `mcc`, `combined`, `scalarized`, `utopia`.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `alpha_grid` | [0.25, 0.5, 0.75, 1.0] | Candidate exponents |
| `inner_cv_folds` | 3 | Folds for inner CV |
| `scoring` | `"geometric_mean"` | Optimisation target |
| Plus all `KNNFairRank` parameters | — | — |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§7`*

KNNFairRankCV is one of the **best-balanced variants**, appearing in the top tier of the final benchmark:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.826 | 2nd |
| MCC | 0.560 | 4th |
| F1 | 0.590 | 3rd |
| ROC-AUC | 0.867 | 3rd |

The $\alpha$ tuning directly addresses the over-correction problem of base v3. Datasets where the Poisson density assumption holds well select $\alpha \approx 1$, recovering v3. Datasets with clustered minorities or uneven majority density select $\alpha < 1$, dampening the correction and reducing false positives. This is data-driven regularisation of the rank-correction strength.

In the Wilcoxon tests (`entrega_final §7`), KNNFairRankCV is significantly better than base KNNFairRank on MCC and F1, while maintaining equivalent G-Mean. The $\alpha$ tuning recovers precision without sacrificing recall — exactly what was needed.

The trade-off is the inner-CV overhead at fit time. On the degenerate subset (very few minority samples, `entrega_final §8`), the inner CV becomes unreliable, and KNNFairRankCV loses some of its advantage — KNNWeighted outperforms it in that regime.

### 3.4 KNNFairRankMagnitudeCV (Modification BC)

**File:** `src/algorithms/fair_rank/core/knn_fair_rank_bc.py`  
**Class:** `KNNFairRankMagnitudeCV`

#### How it works

Combines Modification B (continuous confidence scores) with Modification C (inner-CV $\alpha$ tuning). The inner CV uses magnitude-aware voting when evaluating $\alpha$ — using binary votes to select $\alpha$ then deploying with soft scores would be inconsistent.

$$k_{eff} = \text{round}(r^{\alpha^*}), \quad s_i = \frac{d_{i \cdot k_{eff}}^{maj}}{d_i^{min} + d_{i \cdot k_{eff}}^{maj}}$$

#### Note on results

Not included in the final benchmark in `entrega_final.ipynb`. The combination did not produce results significantly different from KNNFairRankCV alone — getting $\alpha$ right (Mod C) is the dominant improvement, and the continuous-score refinement (Mod B) is secondary. KNNFairRankJointCV (section 3.9), which tunes both $\alpha$ and $n_{votes}$ jointly, achieves better results than either modification in isolation.

### 3.5 KNNFairRankLocalOdds (Modification E)

**File:** `src/algorithms/fair_rank/local/knn_fair_rank_local_odds.py`  
**Class:** `KNNFairRankLocalOdds`

#### How it works

Replaces the global $k_{eff} = r$ with a **per-query** estimate of the local density ratio, using a counting trick that avoids dimensionality estimation.

For each minority rank $i$, count how many majority points are closer to $x$ than $d_i^{min}$:

$$j_i = \texttt{searchsorted}(d^{maj},\; d_i^{min})$$

Under homogeneous Poisson, $E[j_i / i] = r$. The median of these estimates is then Bayesian-shrunk toward the global $r$:

$$k_{eff}(x) = \text{round}\left(\frac{\text{median}_i(j_i/i) + \lambda r}{1 + \lambda}\right)$$

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_probe` | `k_min` | Minority ranks used to estimate local ratio |
| `shrinkage` | 1.0 | Prior weight $\lambda$ toward global $r$ |

#### Note on results

Not included in the final benchmark. The local estimation is theoretically sound but sensitive to the shrinkage hyperparameter $\lambda$: with $\lambda = 1$, the local estimate is heavily pulled toward the global $r$, nearly recovering v3; with small $\lambda$, the noisy per-point count $j_i/i$ dominates. Finding the right $\lambda$ per dataset requires its own CV, at which point KNNFairRankCV achieves the same goal with less complexity.

### 3.6 KNNFairRankEnsemble

**File:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_ens.py`  
**Class:** `KNNFairRankEnsemble`

#### How it works

Instead of selecting a single $(n_{votes}, \alpha)$ pair, sweeps the **full grid** at inference time. For every $(n_v, \alpha) \in n_{votes\_grid} \times \alpha_{grid}$, run binary votes with $k_{eff} = \text{round}(r^\alpha)$. All votes are pooled:

$$\text{vote fraction} = \frac{\text{total minority wins across all } (n_v, \alpha) \text{ pairs}}{\text{total votes cast}}$$

Default grids: $\alpha \in \{0.25, 0.5, 0.75, 1.0\}$, $n_v \in \{1, 2, 3, 5, 7, 10\}$.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `alpha_grid` | [0.25, 0.5, 0.75, 1.0] | Correction exponent candidates |
| `n_votes_grid` | [1, 2, 3, 5, 7, 10] | Vote count candidates |
| Plus `k_min`, `k_maj_*` from `KNNFairRank` | — | — |

#### Results and analysis

> *Source: `entrega_final.ipynb §6, §11.1`*

KNNFairRankEnsemble is the **ranking metric specialist** of the benchmark:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.772 | 9th |
| MCC | **0.582** | **1st** |
| F1 | **0.614** | **1st** |
| ROC-AUC | **0.884** | **1st** |
| PR-AUC | **0.649** | **1st** |

From `entrega_final §11.1`:

> *"KNNFairRankEnsemble ranks #1 on both [ROC-AUC and PR-AUC] (ROC-AUC = 0.878, PR-AUC = 0.649). It is **statistically significantly better than SMOTE+KNN** on both."*

And from the MCC footprint analysis (`entrega_final §9.2`):

> *"KNNFairRankEnsemble wins a further 28% [of datasets on MCC]."*

The mechanism is clear: the grid includes $\alpha \in \{0.25, 0.5\}$, which use $k_{eff} < r$ — conservative corrections that vote minority less aggressively. These conservative votes cancel out the aggressive $\alpha = 1$ votes, reducing the false-positive rate and boosting precision. MCC, F1, and ROC-AUC all reward this balance.

The G-Mean drops to 9th because the same conservative $\alpha$ values reduce minority recall: when some votes say "minority" and others say "majority", the pooled fraction lands closer to 0.5, and borderline cases get classified as majority. This is the exact trade-off: the ensemble is optimal for precision-recall balance but not for maximising minority recall.

**When to prefer this algorithm:** when ROC-AUC or PR-AUC is the primary evaluation metric, or when MCC/F1 is prioritised over G-Mean.

### 3.7 KNNFairRankTopoJoint

**File:** `src/algorithms/fair_rank/topology/knn_fair_rank_topo_joint.py`  
**Class:** `KNNFairRankTopoJoint`

#### How it works

Partitions the training space into regions via Ward linkage on the joint point cloud, then assigns a per-region $k_{eff}$ based on the local imbalance ratio.

**At fit time:**
1. Build a Ward dendrogram on all $N$ training points.
2. Find the cut with the largest relative gap (gap / cloud diameter) where every region has $\geq$ `min_region_samples` points. Fall back to one global region if no valid cut exists.
3. For each region $R_j$: $k_{eff}(R_j) = \min\left(\frac{N_{maj}(R_j) + \lambda}{N_{min}(R_j) + \lambda},\ r\right)$

**At predict time:** assign $x$ to the region of its nearest training point, use that region's $k_{eff}$.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `min_persistence_ratio` | 0.05 | Minimum gap/diameter for a split |
| `laplace_smooth` | 1.0 | $\lambda$ in per-region $k_{eff}$ |
| `min_region_samples` | adaptive | Min points per region |

#### Results and analysis

> *Source: `entrega_final.ipynb §6`*

KNNFairRankTopoJoint achieves **consistently mid-tier performance** across all metrics:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.821 | 5th |
| MCC | 0.543 | 7th |
| ROC-AUC | 0.862 | 6th |

It never leads any metric, nor is it last. This reflects the nature of the topology approach: the Ward clustering finds meaningful spatial partitions when the minority class genuinely occupies distinct sub-regions of the feature space. On those datasets, the per-region $k_{eff}$ is more appropriate than the global $r$, and performance improves. On datasets where minority points are uniformly distributed throughout the space, the algorithm falls back to a single global region, recovering base v3. Across a diverse 40-dataset benchmark these two behaviours average out to a middle ranking.

The unique advantage of this variant is **interpretability**: the `zone_counts_` attribute classifies training regions into majority-dominated, minority-dominated, and mixed zones, and `point_k_eff_` gives a per-training-point correction strength — useful for understanding where in the feature space the imbalance is most severe.

### 3.8 KNNFairRankJackknife

**File:** `src/algorithms/fair_rank/resampling/knn_fair_rank_jackknife.py`  
**Class:** `KNNFairRankJackknife`

#### How it works

A Leave-One-Out jackknife over the minority distance sequence at each query:

For $j = 1, \ldots, k_{probe}$:
1. Remove $d_j^{min}$ from the minority distance array.
2. Run the standard v3 binary-vote step on the remaining sequence.

Final vote fraction: $\bar{f} = \frac{1}{k_{probe}} \sum_j f_j$. Predict minority iff $\bar{f} > 0.5$.

**Motivation:** In v3, a single anomalous minority neighbour (duplicate, mislabelled point, or sampling artefact) sitting very close to $x$ dominates one comparison and can flip the decision. The LOO average dilutes its influence: if removing it changes the vote fraction substantially, the average is less extreme and the prediction is more conservative.

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_probe` | `n_votes` | Number of LOO trials (auto-capped at `k_min - n_votes`) |

#### Results and analysis

> *Source: `entrega_final.ipynb §6–§7`*

KNNFairRankJackknife is one of the most **consistently strong** variants in the benchmark:

| Metric | Score | Rank |
|---|---|---|
| G-Mean | 0.819 | 6th |
| MCC | 0.530 | 6th |
| ROC-AUC | **0.873** | **2nd** |

The Wilcoxon tests show it produces some of the strongest improvements over base KNNFairRank on MCC — the LOO averaging eliminates the systematic misclassification caused by anomalous minority points that v3 is vulnerable to.

The strong ROC-AUC (2nd) comes from the same mechanism: the average of jackknife vote fractions produces better-calibrated soft scores than the single-pass binary vote fraction. When a noisy minority point pulls the vote fraction to 0.8 but its removal drops it to 0.6, the jackknife average is 0.68 — the uncertainty is correctly reflected. The ROC-AUC scorer benefits directly from this calibration.

The trade-off is $O(k_{probe})$ additional vote computations per query — more expensive than v3 but cheaper than variants requiring inner CV at fit time. It is also immune to the degenerate dataset problem (no inner CV needed), making it reliable across a wider range of dataset sizes.

### 3.9 KNNFairRankJointCV

**File:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_joint_cv.py`  
**Class:** `KNNFairRankJointCV`

#### How it works

Extends KNNFairRankCV (Mod C) to jointly tune both $\alpha$ **and** $n_{votes}$ via inner stratified CV. Rather than fixing $n_{votes}$ and optimising $\alpha$ alone, it searches over the full $(n_{votes}, \alpha)$ grid and selects the pair that maximises the scoring metric on held-out folds.

#### Results and analysis

> *Source: `entrega_final.ipynb §6, §9, §11–§12` and `meta_analysis.ipynb §6–§7`*

KNNFairRankJointCV is the **overall winner** of the benchmark — the algorithm that best reconciles G-Mean and MCC.

| Metric | Score | Rank |
|---|---|---|
| G-Mean | **0.828** | **1st** |
| MCC | 0.574 | 2nd |
| F1 | 0.606 | 2nd |
| ROC-AUC | 0.863 | 4th |

From `entrega_final §11.1`:

> *"KNNFairRankJointCV ranks #1 (mean G-Mean = 0.796 vs 0.631 for KNNOptK). It is not significantly better than SMOTE+KNN after Holm correction with 9 comparisons (p_corr = 0.136, Δ = +0.030), but leads the ranking consistently."*

The instance space analysis (`entrega_final §9.3`) provides the strongest in-context validation:

> *"KNNFairRankJointCV is the only algorithm whose G-Mean footprint grows from Q1 (20%) to Q4 (30%). Its relative advantage increases as imbalance becomes more severe — consistent with the theoretical prediction that the fair-rank correction $\alpha$ should scale with IR."*

This is the key result: **the algorithm designed for high imbalance wins proportionally more often precisely where it should**.

The meta-analysis notebook (`meta_analysis.ipynb §6–§7`) adds a predictive dimension. A Leave-One-Dataset-Out (LODO-CV) binary meta-classifier — trained on dataset meta-features to predict whether FairRankJointCV will beat KNNOptK — achieves above-baseline accuracy, and the FairRank win rate along the IR axis (stress test) shows a monotone increase from mild to severe imbalance quartiles with Wilson confidence intervals that separate clearly from 50%.

**Why joint tuning beats separate tuning:** $n_{votes}$ and $\alpha$ interact. With high $n_{votes}$ and low $\alpha$, you get many conservative votes. With low $n_{votes}$ and high $\alpha$, you get few aggressive votes. Optimising one while fixing the other finds a local optimum; the joint search finds the globally best combination for each dataset.

**Limitation in the degenerate regime:** In the subset of 9 datasets with extremely few minority samples (`entrega_final §8`), KNNFairRankJointCV drops to 4th because the inner CV that selects $(n_{votes}, \alpha)$ is unreliable with so few minority examples per fold. KNNWeighted, which requires no optimisation, outperforms it in this regime.

---

## 4. Overall Comparison

### Mean scores and ranks (5-rep × 10-fold CV, 40 datasets)

Sorted by G-Mean rank:

| Algorithm | G-Mean | MCC | F1 | ROC-AUC | G-Mean rank | MCC rank |
|---|---|---|---|---|---|---|
| **KNNFairRankJointCV** | **0.828** | 0.574 | 0.606 | 0.863 | **1** | 2 |
| KNNFairRankCV | 0.826 | 0.560 | 0.590 | 0.867 | 2 | 4 |
| KNNFairRankOptVotes | 0.828 | 0.537 | 0.568 | 0.857 | 3 | 8 |
| SMOTE+KNN | 0.807 | 0.550 | 0.595 | 0.850 | 4 | 5 |
| KNNFairRankTopoJointBootstrap | 0.821 | 0.543 | 0.577 | 0.862 | 5 | 7 |
| KNNFairRankJackknife | 0.819 | 0.530 | 0.559 | 0.873 | 6 | 6 |
| KNNWeighted | 0.797 | 0.534 | 0.583 | 0.841 | 7 | 9 |
| KNNFairRank | 0.819 | 0.503 | 0.534 | 0.863 | 7 | 10 |
| **KNNFairRankEnsemble** | 0.772 | **0.582** | **0.614** | **0.884** | 9 | **1** |
| KNNOptK | 0.673 | 0.556 | 0.586 | 0.803 | 10 | 3 |

### The central trade-off (`entrega_final §11.1`)

> *"No single algorithm dominates all metrics — the results split cleanly by metric type."*

- **Best for G-Mean** (maximise minority recall): `KNNFairRankJointCV`
- **Best for MCC, F1, ROC-AUC, PR-AUC** (balance precision and recall): `KNNFairRankEnsemble`
- **Surprising MCC performer**: `KNNOptK` ranks 3rd on MCC despite being last on G-Mean — a conservative baseline that achieves high specificity by rarely predicting minority
- **Degenerate datasets** (very few minority samples): `KNNWeighted` — no inner CV dependency

### MCC pathology (`entrega_final §9.2`)

> *"KNNOptK wins MCC on 17/40 datasets (42%) despite ranking last on G-Mean. This pattern confirms that MCC alone is an unreliable criterion for imbalanced classification and should not be used as the sole evaluation metric."*

### G-Mean grows with imbalance severity (`entrega_final §9.3`)

KNNFairRankJointCV is the **only algorithm whose G-Mean win rate increases from mild (Q1: 20%) to severe imbalance (Q4: 30%)**. All other algorithms either stay flat or degrade. This confirms the theoretical prediction: the fair-rank correction $k_{eff} = r^\alpha$ becomes more important — and more accurate — as the imbalance ratio grows.

---

*This document was generated from source code in `src/algorithms/`. Results and interpretations are sourced directly from `notebooks/entrega_final.ipynb` and `notebooks/meta_analysis.ipynb`.*